# MRC Nucleus — Batch Normal-Vector Reslice (V6)

**Workflow:**
1. Set the GUI backend and import libraries.
2. Configure paths and reslice settings.
3. Inspect the MRC volume header.
4. Load and validate the surface-points spreadsheet.
5. Define helper functions.
6. Run the **batch loop** — for each point in the spreadsheet:
   - Show a reference pop-up (XY/XZ/YZ orthogonal slices).
   - Draw tangent lines interactively (one plane at a time).
   - Compute the surface normal and save a text report.
   - Reslice along n, u, and v and save three MRC files.

---
### Dependencies
```
pip install mrcfile numpy scipy matplotlib tifffile Pillow openpyxl pandas
```
Requires a desktop GUI toolkit: Tk (bundled with most Python distributions),
Qt5 (`pip install PyQt5`), or Qt6 (`pip install PyQt6`).

> **Important:** Restart the kernel before running Cell 1 if any other
> Matplotlib code has already been run in this session.

---
## Cell 1 — Backend and imports

In [ ]:
import matplotlib

def _find_gui_backend():
    for backend in ('TkAgg', 'Qt5Agg', 'Qt6Agg', 'wxAgg', 'MacOSX'):
        try:
            matplotlib.use(backend)
            import matplotlib.pyplot as _p
            _f = _p.figure()
            _p.close(_f)
            print(f'GUI backend: {backend} OK')
            return
        except Exception as exc:
            print(f'  {backend}: {exc}')
    raise RuntimeError('No GUI backend found. Try: pip install PyQt5')

_find_gui_backend()

import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import mrcfile
import openpyxl
from scipy.ndimage import map_coordinates
from pathlib import Path
from datetime import datetime

try:
    import tifffile
    _HAVE_TIFFFILE = True
except ImportError:
    _HAVE_TIFFFILE = False
    print('tifffile not found -- PIL will be used for TIFF saving.')

print('Imports OK')

---
## Cell 2 — Configuration

In [ ]:
# Source volume
MRC_PATH = 'your_nucleus.mrc'          # <-- path to your MRC file

# Input spreadsheet (see surface_points_template.xlsx)
# Required columns: point_id, label, Z, Y, X
POINTS_XLSX = Path('surface_points_template.xlsx')  # <--

# Output folder -- all outputs written here, named by point_id
#   batch_output/resliced_normal_01.mrc
#   batch_output/resliced_tangent_u_01.mrc
#   batch_output/resliced_tangent_v_01.mrc
#   batch_output/normal_vector_01.txt
#   batch_output/ortho_slices/point_01/  (TIFFs, if enabled)
OUTPUT_DIR = Path('batch_output')

# Reslice extent: total slices = SLICES_BELOW + 1 + SLICES_ABOVE
SLICES_BELOW = 30
SLICES_ABOVE = 30

# Output patch size per resliced plane (pixels)
OUT_HEIGHT = 256
OUT_WIDTH  = 256

# Save XY/XZ/YZ orthogonal slices as TIFFs?
SAVE_ORTHO_TIFFS = True

---
## Cell 3 — Inspect volume header

In [ ]:
with mrcfile.mmap(MRC_PATH, mode='r', permissive=True) as mrc:
    _nz, _ny, _nx = mrc.data.shape
    _dtype        = mrc.data.dtype
    _vox          = mrc.voxel_size

print(f'Volume dimensions : Z={_nz}, Y={_ny}, X={_nx}')
print(f'Data type         : {_dtype}')
print(f'Voxel size        : {_vox} Angstrom')
print(f'Valid ranges  Z: 0-{_nz-1}   Y: 0-{_ny-1}   X: 0-{_nx-1}')

---
## Cell 4 — Load and validate the points spreadsheet

Reads the spreadsheet, validates every coordinate against the volume
bounds, and prints a summary table.

In [ ]:
def load_points(xlsx_path, nz, ny, nx):
    """
    Read the surface-points spreadsheet using pandas.

    pandas handles header detection, type coercion, and NaN handling cleanly.
    openpyxl is kept only for the status write-back in update_status().

    The spreadsheet has a title row, a column-header row, a sub-header notes
    row, then data rows.  We locate the header row by scanning for 'point_id'
    in column A, then pass that row index to pd.read_excel() as `header`.

    Returns a list of dicts: {point_id, label, Z, Y, X, notes, xlsx_row}
    where xlsx_row is the 1-based Excel row number used by openpyxl write-back.
    """
    # ── Find which row holds the column headers ────────────────────────────
    # Read without a header first so we can scan column A for 'point_id'.
    raw = pd.read_excel(str(xlsx_path), sheet_name='Surface Points', header=None)
    header_row_idx = None   # 0-based pandas row index
    for i, val in enumerate(raw.iloc[:, 0]):
        if str(val).strip() == 'point_id':
            header_row_idx = i
            break
    if header_row_idx is None:
        raise ValueError("Cannot find 'point_id' header in column A.")

    # ── Read the data using that header row, skipping the notes sub-header ─
    # skiprows skips the notes row immediately after the header.
    df = pd.read_excel(
        str(xlsx_path),
        sheet_name='Surface Points',
        header=header_row_idx,
        skiprows=[header_row_idx + 1],  # skip the sub-header notes row
        dtype={'point_id': str},        # keep IDs as strings ('01', '02', ...)
    )

    for col in ('point_id', 'Z', 'Y', 'X'):
        if col not in df.columns:
            raise ValueError(f"Required column '{col}' not found in spreadsheet.")

    # Drop entirely blank rows
    df = df.dropna(subset=['point_id'])

    # Coerce coordinates to integer (Excel often stores them as float)
    for col in ('Z', 'Y', 'X'):
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

    # Fill optional columns
    df['label'] = df['label'].fillna('').astype(str).str.strip()
    df['notes'] = df['notes'].fillna('').astype(str).str.strip()

    # Drop placeholder rows (Z==Y==X==0)
    df = df[~((df['Z'] == 0) & (df['Y'] == 0) & (df['X'] == 0))].reset_index(drop=True)

    # ── Validate coordinates ───────────────────────────────────────────────
    # xlsx_row: openpyxl 1-based row = pandas 0-based original index
    #   + header_row_idx (header row)
    #   + 1 (notes sub-header row we skipped)
    #   + 2 (1 to convert 0-based to 1-based, 1 for the header row itself)
    # Simpler to recompute: header is at Excel row (header_row_idx + 1),
    # notes row is (header_row_idx + 2), data starts at (header_row_idx + 3).
    errors = []
    points = []
    for df_idx, row in df.iterrows():
        pid = str(row['point_id']).strip()
        z, y, x = int(row['Z']), int(row['Y']), int(row['X'])
        xlsx_row = header_row_idx + 3 + df_idx   # 1-based Excel row for write-back

        bad = []
        if not (0 <= z < nz): bad.append(f'Z={z} outside [0,{nz-1}]')
        if not (0 <= y < ny): bad.append(f'Y={y} outside [0,{ny-1}]')
        if not (0 <= x < nx): bad.append(f'X={x} outside [0,{nx-1}]')
        if bad:
            errors.append(f'  Row {xlsx_row} (id={pid}): ' + ', '.join(bad))
            continue

        points.append({
            'point_id': pid,
            'label':    str(row['label']),
            'Z': z, 'Y': y, 'X': x,
            'notes':    str(row['notes']),
            'xlsx_row': xlsx_row,
        })

    if errors:
        raise ValueError('Spreadsheet errors:\n' + '\n'.join(errors))
    return points


_points = load_points(POINTS_XLSX, _nz, _ny, _nx)

print(f'Spreadsheet : {POINTS_XLSX}')
print(f'Points loaded: {len(_points)}')
print()
print(f"  {'ID':>4}  {'Label':<22}  {'Z':>6}  {'Y':>6}  {'X':>6}  Notes")
print('  ' + '-' * 70)
for p in _points:
    print(f"  {p['point_id']:>4}  {p['label']:<22}  "
          f"{p['Z']:>6}  {p['Y']:>6}  {p['X']:>6}  {p['notes']}")
print()
print('All coordinates valid.  Run Cell 5 to define helper functions.')

---
## Cell 5 — Helper functions

Run once to define all classes and functions used in Cell 6.

In [ ]:
# ─── display utility ────────────────────────────────────────────────────────

def _norm_display(arr):
    lo, hi = np.percentile(arr, [1, 99])
    return np.clip((arr - lo) / (hi - lo + 1e-9), 0, 1).astype(np.float32)


# ─── ortho slice extraction ──────────────────────────────────────────────────

def extract_ortho_slices(mrc_path, z, y, x):
    with mrcfile.mmap(mrc_path, mode='r', permissive=True) as mrc:
        xy = np.array(mrc.data[z, :, : ], dtype=np.float32)
        xz = np.array(mrc.data[:, y, : ], dtype=np.float32)
        yz = np.array(mrc.data[:, :, x ], dtype=np.float32)
    return xy, xz, yz


def _save_tiff(arr, path):
    if _HAVE_TIFFFILE:
        tifffile.imwrite(str(path), arr)
    else:
        from PIL import Image
        Image.fromarray(arr).save(str(path))


def show_reference_popup(xy, xz, yz, z, y, x, point_id, label):
    specs = [
        (xy, f'XY  (Z={z})', 'X (px)', 'Y (px)', x, y),
        (xz, f'XZ  (Y={y})', 'X (px)', 'Z (px)', x, z),
        (yz, f'YZ  (X={x})', 'Y (px)', 'Z (px)', y, z),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    lbl_str = f'  [{label}]' if label else ''
    fig.canvas.manager.set_window_title(f'Reference -- Point {point_id}{lbl_str}')
    for ax, (img, title, h_lbl, v_lbl, cx, cy) in zip(axes, specs):
        ax.imshow(_norm_display(img), cmap='gray', origin='lower', aspect='auto')
        ax.axhline(cy, color='red', lw=0.9, alpha=0.7)
        ax.axvline(cx, color='red', lw=0.9, alpha=0.7)
        ax.plot(cx, cy, 'r+', ms=12, mew=1.5)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel(h_lbl, fontsize=9)
        ax.set_ylabel(v_lbl, fontsize=9)
    fig.suptitle(
        f'Point {point_id}{lbl_str}   Z={z}, Y={y}, X={x}   '
        '(close window to begin tangent picking)', fontsize=11)
    fig.tight_layout()
    plt.show(block=True)
    plt.pause(0.2)   # flush GUI events so the window fully closes
    plt.close('all') # destroy the figure before the next window opens

class TangentPicker:
    def __init__(self, image, title, h_label, v_label, crosshair):
        self.image     = image
        self.title     = title
        self.h_label   = h_label
        self.v_label   = v_label
        self.crosshair = crosshair
        self.points    = []
        self._line     = None
        self._dots     = []
        self.ready     = False
        self.result    = None

    def run(self):
        self._build_figure()
        plt.show(block=True)
        plt.pause(0.2)      # flush GUI events so the window fully closes
        plt.close('all')    # destroy the figure before the next window opens
        if not self.ready:
            raise RuntimeError(
                f"Window '{self.title}' closed before two points were placed. "
                'Re-run Cell 6 to restart this point.')
        return self.result

    def _build_figure(self):
        self.fig, self.ax = plt.subplots(1, 1, figsize=(9, 7))
        self.fig.canvas.manager.set_window_title(f'{self.title} -- draw tangent line')
        plt.subplots_adjust(bottom=0.13, top=0.88)
        self.ax.imshow(_norm_display(self.image), cmap='gray',
                       origin='lower', aspect='auto')
        cx, cy = self.crosshair
        self.ax.axhline(cy, color='red', lw=0.9, alpha=0.6)
        self.ax.axvline(cx, color='red', lw=0.9, alpha=0.6)
        self.ax.plot(cx, cy, 'r+', ms=14, mew=1.5, zorder=5)
        self.ax.set_xlabel(self.h_label, fontsize=10)
        self.ax.set_ylabel(self.v_label, fontsize=10)
        self._status = self.fig.text(
            0.5, 0.02,
            'Left-click 2 pts on surface  |  Right-click to clear  |  '
            'Use toolbar zoom/pan to navigate, then click pointer icon to resume picking',
            ha='center', va='bottom', fontsize=9, color='#333333')
        self._update_title()
        self.fig.canvas.mpl_connect('button_press_event', self._on_click)

    def _on_click(self, event):
        if event.inaxes is not self.ax:
            return
        # Do nothing while the toolbar zoom or pan tool is active —
        # let Matplotlib handle those clicks for navigation instead.
        if self.fig.canvas.toolbar and self.fig.canvas.toolbar.mode != '':
            return
        if event.button == 3:
            self._clear_picks()
            self.fig.canvas.draw_idle()
            return
        if event.button == 1 and not self.ready:
            h, v = event.xdata, event.ydata
            self.points.append((h, v))
            dot, = self.ax.plot(h, v, 'yo', ms=9, zorder=7)
            self._dots.append(dot)
            if len(self.points) == 1:
                self._status.set_text(f'Point 1 at ({h:.1f},{v:.1f}) -- click a second point')
            elif len(self.points) == 2:
                self._draw_tangent_line()
                self._finalise()
            self.fig.canvas.draw_idle()

    def _draw_tangent_line(self):
        (x0, y0), (x1, y1) = self.points
        W, H = self.image.shape[1] - 1, self.image.shape[0] - 1
        if abs(x1 - x0) < 1e-9:
            xs, ys = [x0, x0], [0, H]
        else:
            m  = (y1 - y0) / (x1 - x0)
            xs = [0, W]
            ys = [y0 + m*(0 - x0), y0 + m*(W - x0)]
        self._line, = self.ax.plot(xs, ys, color='gold', lw=2.0, alpha=0.9, zorder=6)

    def _clear_picks(self):
        self.points = []; self.ready = False; self.result = None
        if self._line:
            self._line.remove(); self._line = None
        for d in self._dots: d.remove()
        self._dots = []
        self._status.set_text('Cleared -- left-click 2 points  |  Right-click to clear again')
        self._update_title()

    def _finalise(self):
        self.result = self._lift(*self._raw_tangent())
        self.ready  = True
        (x0, y0), (x1, y1) = self.points
        self._status.set_text(
            f'Tangent set: ({x0:.1f},{y0:.1f}) -> ({x1:.1f},{y1:.1f})  |  Close window to continue')
        self._status.set_color('green')
        self._update_title(done=True)

    def _update_title(self, done=False):
        if done:
            self.ax.set_title(
                self.title + '  [done] -- close window to continue',
                fontsize=11, color='green')
        else:
            self.ax.set_title(
                self.title + '  -- click 2 points on the nuclear surface',
                fontsize=11, color='black')

    def _raw_tangent(self):
        (h0, v0), (h1, v1) = self.points
        return h1 - h0, v1 - v0

    def _lift(self, dh, dv): raise NotImplementedError


class XYPicker(TangentPicker):
    def _lift(self, dh, dv):
        v = np.array([0.0, dv, dh])
        return v / (np.linalg.norm(v) + 1e-12)

class XZPicker(TangentPicker):
    def _lift(self, dh, dv):
        v = np.array([dv, 0.0, dh])
        return v / (np.linalg.norm(v) + 1e-12)

class YZPicker(TangentPicker):
    def _lift(self, dh, dv):
        v = np.array([dv, dh, 0.0])
        return v / (np.linalg.norm(v) + 1e-12)


# ─── normal computation ──────────────────────────────────────────────────────

def compute_normal(tangents_3d):
    T = np.stack(tangents_3d, axis=0)
    _, s, Vt = np.linalg.svd(T)
    n = Vt[-1]
    if n[0] < 0: n = -n
    cond = s[0] / (s[-1] + 1e-12)
    return n, s, cond


# ─── normal vector report ────────────────────────────────────────────────────

def save_normal_report(path, origin_zyx, normal_zyx, tangents, svals, cond,
                        mrc_path, point_id, label, notes):
    lines = [
        'MRC Normal-Vector Report',
        '=' * 50,
        f'Generated  : {datetime.now().strftime("%Y-%m-%d  %H:%M:%S")}',
        f'Source MRC : {mrc_path}',
        f'Point ID   : {point_id}',
        f'Label      : {label}',
        f'Notes      : {notes}',
        '',
        'Surface point (voxel coordinates)',
        f'  Z = {origin_zyx[0]}',
        f'  Y = {origin_zyx[1]}',
        f'  X = {origin_zyx[2]}',
        '',
        '3-D tangent vectors (Z, Y, X)',
        f'  T_XY = [{tangents[0][0]:.8f},  {tangents[0][1]:.8f},  {tangents[0][2]:.8f}]',
        f'  T_XZ = [{tangents[1][0]:.8f},  {tangents[1][1]:.8f},  {tangents[1][2]:.8f}]',
        f'  T_YZ = [{tangents[2][0]:.8f},  {tangents[2][1]:.8f},  {tangents[2][2]:.8f}]',
        '',
        'SVD singular values',
        f'  [{svals[0]:.6f},  {svals[1]:.6f},  {svals[2]:.6f}]',
        f'  Condition number : {cond:.2f}' +
            ('  HIGH -- consider redrawing a tangent' if cond > 100 else '  well-determined'),
        '',
        'Surface normal n (Z, Y, X) -- unit vector',
        f'  nZ = {normal_zyx[0]:.10f}',
        f'  nY = {normal_zyx[1]:.10f}',
        f'  nX = {normal_zyx[2]:.10f}',
        '',
        'Polar angles (degrees)',
        f'  Elevation from XY plane : '
            f'{np.degrees(np.arcsin(np.clip(normal_zyx[0], -1, 1))):.4f} deg',
        f'  Azimuth in XY plane     : '
            f'{np.degrees(np.arctan2(normal_zyx[1], normal_zyx[2])):.4f} deg',
        '=' * 50,
    ]
    Path(path).write_text('\n'.join(lines))


# ─── geometry ────────────────────────────────────────────────────────────────

def build_inplane_axes(normal_zyx):
    n   = normal_zyx / np.linalg.norm(normal_zyx)
    ref = np.array([1.0, 0.0, 0.0])
    if abs(np.dot(n, ref)) > 0.9:
        ref = np.array([0.0, 1.0, 0.0])
    u = ref - np.dot(ref, n) * n
    u /= np.linalg.norm(u)
    v  = np.cross(n, u)
    v /= np.linalg.norm(v)
    return u, v


# ─── subvolume crop ───────────────────────────────────────────────────────────

def _crop_subvolume(mrc_path, origin, n_hat, u, v,
                    slices_below, slices_above, out_height, out_width, pad=4):
    half_h, half_w = (out_height - 1) / 2.0, (out_width - 1) / 2.0
    corners = []
    for s in (-slices_below, slices_above):
        ctr = origin + s * n_hat
        for sv in (-half_h, half_h):
            for su in (-half_w, half_w):
                corners.append(ctr + sv * v + su * u)
    corners = np.array(corners)
    with mrcfile.mmap(mrc_path, mode='r', permissive=True) as mrc:
        vol_shape = np.array(mrc.data.shape)
        lo = np.clip(np.floor(corners.min(axis=0)).astype(int) - pad, 0, vol_shape - 1)
        hi = np.clip(np.ceil( corners.max(axis=0)).astype(int) + pad + 1, 1, vol_shape)
        z0, y0, x0 = lo
        z1, y1, x1 = hi
        mb = (z1-z0) * (y1-y0) * (x1-x0) * 4 / 1e6
        print(f'    Crop Z[{z0}:{z1}] Y[{y0}:{y1}] X[{x0}:{x1}]  '
              f'({z1-z0}x{y1-y0}x{x1-x0} vox, {mb:.1f} MB)', end='', flush=True)
        subvol = np.array(mrc.data[z0:z1, y0:y1, x0:x1], dtype=np.float32)
        print(' done')
    return subvol, (z0, y0, x0)


# ─── reslice ─────────────────────────────────────────────────────────────────

def reslice_along_normal(mrc_path, origin_zyx, normal_zyx,
                          slices_below, slices_above, out_height, out_width):
    import time, sys
    from IPython.display import clear_output as _clear

    origin = np.asarray(origin_zyx, dtype=float)
    n_hat  = np.asarray(normal_zyx, dtype=float)
    n_hat /= np.linalg.norm(n_hat)
    u, v   = build_inplane_axes(n_hat)

    subvol, (cz0, cy0, cx0) = _crop_subvolume(
        mrc_path, origin, n_hat, u, v,
        slices_below, slices_above, out_height, out_width)

    cols = np.linspace(-(out_width -1)/2.0, (out_width -1)/2.0, out_width)
    rows = np.linspace(-(out_height-1)/2.0, (out_height-1)/2.0, out_height)
    gc, gr = np.meshgrid(cols, rows)

    offsets  = list(range(-slices_below, slices_above + 1))
    n_slices = len(offsets)
    stack    = np.zeros((n_slices, out_height, out_width), dtype=np.float32)
    BAR_W    = 36

    def _prog(done, total, t0):
        frac = done / total
        bar  = chr(9608)*int(BAR_W*frac) + chr(9617)*(BAR_W-int(BAR_W*frac))
        el   = time.time() - t0
        eta  = (f'{int(el/done*(total-done)//60):02d}:{int(el/done*(total-done)%60):02d}'
                if done > 0 else '--:--')
        _clear(wait=True)
        print(f'    [{bar}] {done:>{len(str(total))}}/{total}  '
              f'({frac*100:5.1f}%)  '
              f'elapsed {int(el//60):02d}:{int(el%60):02d}  ETA {eta}')
        sys.stdout.flush()

    t0 = time.time()
    _prog(0, n_slices, t0)
    for idx, s in enumerate(offsets):
        ctr = origin + s * n_hat
        cz  = ctr[0] + gr*v[0] + gc*u[0]
        cy  = ctr[1] + gr*v[1] + gc*u[1]
        cx  = ctr[2] + gr*v[2] + gc*u[2]
        coords  = np.stack([(cz-cz0).ravel(), (cy-cy0).ravel(), (cx-cx0).ravel()])
        sampled = map_coordinates(subvol, coords, order=1,
                                   mode='constant', cval=0.0, prefilter=False)
        stack[idx] = sampled.reshape(out_height, out_width)
        _prog(idx+1, n_slices, t0)
    return stack, u, v


# ─── save MRC ────────────────────────────────────────────────────────────────

def save_resliced_mrc(stack, output_path, source_mrc_path,
                       origin_zyx, normal_zyx, u_axis, v_axis,
                       slices_below, description, point_id, label):
    with mrcfile.mmap(source_mrc_path, mode='r', permissive=True) as src:
        vox_x = float(src.voxel_size.x)
    with mrcfile.new(str(output_path), overwrite=True) as mrc:
        mrc.set_data(stack)
        mrc.voxel_size = vox_x
        oz, oy, ox = origin_zyx
        nz, ny, nx = np.round(normal_zyx, 6)
        mrc.add_label(f'Point {point_id} {label} -- {description}')
        mrc.add_label(f'Origin voxel Z={oz} Y={oy} X={ox}')
        mrc.add_label(f'Normal (Z,Y,X)={nz:.6f} {ny:.6f} {nx:.6f}')
        mrc.add_label(f'U-axis (Z,Y,X)={u_axis[0]:.6f} {u_axis[1]:.6f} {u_axis[2]:.6f}')
        mrc.add_label(f'V-axis (Z,Y,X)={v_axis[0]:.6f} {v_axis[1]:.6f} {v_axis[2]:.6f}')
        mrc.add_label(f'Slice 0 is {slices_below} steps below surface')
    sz = Path(output_path).stat().st_size
    print(f'    Saved: {Path(output_path).name}  ({sz/1e6:.1f} MB)')


# ─── spreadsheet status update ───────────────────────────────────────────────
# openpyxl is used only here, for surgical cell writes.
# pandas (read-only) and openpyxl (write) are kept separate so that
# reformatting/data loss cannot occur during the batch run.

def update_status(xlsx_path, xlsx_row, status):
    """
    Write `status` into the 'status' column of the given Excel row and save.
    Opens and closes the workbook each call to avoid holding a file lock
    across the entire batch run.
    """
    wb = openpyxl.load_workbook(str(xlsx_path))
    ws = wb['Surface Points']
    # Find the 'status' column index by scanning the header row
    status_col = None
    for cell in ws[1:ws.max_row]:
        for c in cell:
            if c.value == 'status':
                status_col = c.column
                break
        if status_col:
            break
    if status_col is None:
        return   # no status column — skip silently
    ws.cell(row=xlsx_row, column=status_col, value=status)
    wb.save(str(xlsx_path))
    wb.close()


print('Helper functions defined.')

---
## Cell 6 — Batch processing loop

Iterates through every point in the spreadsheet.  For each point:

1. Extracts orthogonal slices and shows a **reference pop-up** — close it when ready.
2. Opens three sequential **tangent-picking pop-ups** (XY then XZ then YZ).
3. Computes the surface normal and saves a text report.
4. Runs three reslices (n, u, v) and saves three MRC files.
5. Marks the row `done` in the spreadsheet and saves it.

If a pop-up window is closed before two points are placed, the batch stops
with an error message.  Fix the issue and re-run — points whose output
files already exist on disk are automatically skipped.

**Output files for each point XX:**
```
batch_output/resliced_normal_XX.mrc
batch_output/resliced_tangent_u_XX.mrc
batch_output/resliced_tangent_v_XX.mrc
batch_output/normal_vector_XX.txt
batch_output/ortho_slices/point_XX/   (TIFFs, if enabled)
```

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
_batch_results = []   # list of per-point result dicts; read by Cell 7

for pt_idx, pt in enumerate(_points):
    pid   = pt['point_id']
    label = pt['label']
    z, y, x = pt['Z'], pt['Y'], pt['X']

    mrc_n = OUTPUT_DIR / f'resliced_normal_{pid}.mrc'
    mrc_u = OUTPUT_DIR / f'resliced_tangent_u_{pid}.mrc'
    mrc_v = OUTPUT_DIR / f'resliced_tangent_v_{pid}.mrc'
    txt   = OUTPUT_DIR / f'normal_vector_{pid}.txt'

    print()
    print('=' * 62)
    lbl_str = f'  [{label}]' if label else ''
    print(f'  POINT {pid}{lbl_str}   ({pt_idx+1}/{len(_points)})')
    print(f'  Z={z},  Y={y},  X={x}')
    print('=' * 62)

    # Skip if all output files already exist
    if all(p.exists() for p in (mrc_n, mrc_u, mrc_v, txt)):
        print('  Output files already exist -- skipping.')
        _batch_results.append({'point_id': pid, 'status': 'skipped',
                                'mrc_n': mrc_n, 'mrc_u': mrc_u, 'mrc_v': mrc_v})
        continue

    try:
        # 1. Ortho slices
        print('  Extracting orthogonal slices ...', end='', flush=True)
        xy_sl, xz_sl, yz_sl = extract_ortho_slices(MRC_PATH, z, y, x)
        print(' done')

        if SAVE_ORTHO_TIFFS:
            tiff_dir = OUTPUT_DIR / 'ortho_slices' / f'point_{pid}'
            tiff_dir.mkdir(parents=True, exist_ok=True)
            _save_tiff(xy_sl, tiff_dir / f'z{z}_y{y}_x{x}_XY.tif')
            _save_tiff(xz_sl, tiff_dir / f'z{z}_y{y}_x{x}_XZ.tif')
            _save_tiff(yz_sl, tiff_dir / f'z{z}_y{y}_x{x}_YZ.tif')
            print(f'  TIFFs saved --> {tiff_dir}')

        # 2. Reference pop-up
        print('  Opening reference window ...')
        show_reference_popup(xy_sl, xz_sl, yz_sl, z, y, x, pid, label)

        # 3. Sequential tangent picking
        plane_defs = [
            (XYPicker, xy_sl, f'[{pid}] XY (Z={z})', 'X (px)', 'Y (px)', (x, y)),
            (XZPicker, xz_sl, f'[{pid}] XZ (Y={y})', 'X (px)', 'Z (px)', (x, z)),
            (YZPicker, yz_sl, f'[{pid}] YZ (X={x})', 'Y (px)', 'Z (px)', (y, z)),
        ]
        tangents = []
        for step, (PickerClass, img, title, h_lbl, v_lbl, xhair) in \
                enumerate(plane_defs, start=1):
            print(f'  [{step}/3]  {title} -- draw tangent, then close window')
            t3d = PickerClass(img, title, h_lbl, v_lbl, xhair).run()
            tangents.append(t3d)
            print(f'    tangent (Z,Y,X) = {np.round(t3d, 5)}')

        # 4. Compute normal
        normal, svals, cond = compute_normal(tangents)
        u_axis, v_axis      = build_inplane_axes(normal)
        print(f'  Normal  n = {np.round(normal, 5)}')
        print(f'  Tangent u = {np.round(u_axis, 5)}')
        print(f'  Tangent v = {np.round(v_axis, 5)}')
        print(f'  Condition number: {cond:.1f}  {"HIGH" if cond > 100 else "OK"}')

        # 5. Save normal report
        save_normal_report(txt, (z, y, x), normal, tangents, svals, cond,
                           MRC_PATH, pid, label, pt['notes'])
        print(f'  Report saved --> {txt.name}')

        # 6. Three reslices
        _rk = dict(mrc_path=MRC_PATH, origin_zyx=(z, y, x),
                   slices_below=SLICES_BELOW, slices_above=SLICES_ABOVE,
                   out_height=OUT_HEIGHT, out_width=OUT_WIDTH)
        _sk = dict(source_mrc_path=MRC_PATH, origin_zyx=(z, y, x),
                   normal_zyx=normal, u_axis=u_axis, v_axis=v_axis,
                   slices_below=SLICES_BELOW, point_id=pid, label=label)

        print('  [1/3]  Reslicing along normal n ...')
        stack_n, _, _ = reslice_along_normal(**_rk, normal_zyx=normal)
        save_resliced_mrc(stack_n, mrc_n, description='along normal n', **_sk)

        print('  [2/3]  Reslicing along tangent u ...')
        stack_u, _, _ = reslice_along_normal(**_rk, normal_zyx=u_axis)
        save_resliced_mrc(stack_u, mrc_u, description='along tangent u', **_sk)

        print('  [3/3]  Reslicing along tangent v ...')
        stack_v, _, _ = reslice_along_normal(**_rk, normal_zyx=v_axis)
        save_resliced_mrc(stack_v, mrc_v, description='along tangent v', **_sk)

        # 7. Mark done in spreadsheet
        update_status(POINTS_XLSX, pt['xlsx_row'], 'done')
        print(f'  Point {pid} complete.')

        _batch_results.append({'point_id': pid, 'status': 'done',
                                'normal': normal, 'u_axis': u_axis, 'v_axis': v_axis,
                                'mrc_n': mrc_n, 'mrc_u': mrc_u, 'mrc_v': mrc_v,
                                'stack_n': stack_n, 'stack_u': stack_u, 'stack_v': stack_v})

    except Exception as exc:
        update_status(POINTS_XLSX, pt['xlsx_row'], 'error')
        print(f'  FAILED point {pid}: {exc}')
        print('  Batch stopped. Fix the issue and re-run Cell 6.')
        raise

print()
print('=' * 62)
done_n    = sum(1 for r in _batch_results if r['status'] == 'done')
skipped_n = sum(1 for r in _batch_results if r['status'] == 'skipped')
print(f'  BATCH COMPLETE -- {done_n} processed, {skipped_n} skipped')
print(f'  Outputs in: {OUTPUT_DIR}/')
print('=' * 62)
print('Batch complete.  Open the output MRC files in your preferred viewer.')